In [18]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .appName("spark-local-app")
        .master("local[*]")
        .getOrCreate()
)

In [19]:
employees = [
    (1, "John", "Doe", 5000.0, "1000", "  usa  ", "+1-555-0199", "987-65-4321"),
    (2, "Jane", "Smith", 6200.0, "", "Canada", "+1-555-0142", "987-65-4322"),
    (3, "Alex", "Jones", 4500.0, "500", "uSa", "+1-555-0173", "987-65-4323"),
    (4, "Emma", "Brown", 7100.0, "abc", "  caNaDa  ", "+1-555-0124", "987-65-4324"),
    (5, "Liam", "Wilson", 4800.0, "", "germany", "+49-30-123456", "987-65-4325"),
    (6, "Olivia", "Taylor", 5300.0, "1200", "GERMANY", "+49-30-654321", "987-65-4326"),
    (7, "Noah", "Davies", 6000.0, " 800 ", "uk", "+44-20-7946-0192", "987-65-4327"),
    (8, "Ava", "Evans", 5500.0, "", "Uk", "+44-20-7946-0111", "987-65-4328"),
    (9, "Ethan", "Thomas", 6700.0, "1500", "FRANCE", "+33-1-4227-7865", "987-65-4329"),
    (10, "Sophia", "Roberts", 5200.0, "not_a_number", "france", "+33-1-4227-1122", "987-65-4330")
]

In [20]:
schema = ["id", "first_name", "last_name", "salary", "bonus", "nationality", "phone_number", "ssn"]

df_employees = spark.createDataFrame(employees, schema=schema)

In [21]:
from pyspark.sql.functions import coalesce, col, initcap, lit, trim

df_employees_cleaned = (
    df_employees
        .withColumn("bonus_cleaned", coalesce(col("bonus").try_cast("int"), lit(0)))
        .withColumn("nationality_cleaned", initcap(trim(col("nationality"))))
        .drop("bonus", "nationality")
)

In [32]:
from pyspark.sql.functions import expr

(
    df_employees
        .withColumn(
            "bonus_cleaned",
            expr("""
                CASE
                    WHEN bonus IS NULL OR bonus = '' OR bonus NOT RLIKE '^[0-9]+$' THEN 0
                    ELSE CAST(bonus AS INT)
                END
            """)
        )
        .show()
)

+---+----------+---------+------+------------+-----------+----------------+-----------+-------------+
| id|first_name|last_name|salary|       bonus|nationality|    phone_number|        ssn|bonus_cleaned|
+---+----------+---------+------+------------+-----------+----------------+-----------+-------------+
|  1|      John|      Doe|5000.0|        1000|      usa  |     +1-555-0199|987-65-4321|         1000|
|  2|      Jane|    Smith|6200.0|            |     Canada|     +1-555-0142|987-65-4322|            0|
|  3|      Alex|    Jones|4500.0|         500|        uSa|     +1-555-0173|987-65-4323|          500|
|  4|      Emma|    Brown|7100.0|         abc|   caNaDa  |     +1-555-0124|987-65-4324|            0|
|  5|      Liam|   Wilson|4800.0|            |    germany|   +49-30-123456|987-65-4325|            0|
|  6|    Olivia|   Taylor|5300.0|        1200|    GERMANY|   +49-30-654321|987-65-4326|         1200|
|  7|      Noah|   Davies|6000.0|        800 |         uk|+44-20-7946-0192|987-65-

In [41]:
from pyspark.sql.functions import when

(
    df_employees
        .withColumn("bonus_cleaned",
                    when(
                        col("bonus").isNull()
                        | (col("bonus") == lit(""))
                        | (~col("bonus").rlike("^[0-9]+$")),
                        0,
                    ).otherwise(col("bonus")))
        .show()
)

+---+----------+---------+------+------------+-----------+----------------+-----------+-------------+
| id|first_name|last_name|salary|       bonus|nationality|    phone_number|        ssn|bonus_cleaned|
+---+----------+---------+------+------------+-----------+----------------+-----------+-------------+
|  1|      John|      Doe|5000.0|        1000|      usa  |     +1-555-0199|987-65-4321|         1000|
|  2|      Jane|    Smith|6200.0|            |     Canada|     +1-555-0142|987-65-4322|            0|
|  3|      Alex|    Jones|4500.0|         500|        uSa|     +1-555-0173|987-65-4323|          500|
|  4|      Emma|    Brown|7100.0|         abc|   caNaDa  |     +1-555-0124|987-65-4324|            0|
|  5|      Liam|   Wilson|4800.0|            |    germany|   +49-30-123456|987-65-4325|            0|
|  6|    Olivia|   Taylor|5300.0|        1200|    GERMANY|   +49-30-654321|987-65-4326|         1200|
|  7|      Noah|   Davies|6000.0|        800 |         uk|+44-20-7946-0192|987-65-

In [42]:
people = [
    (1, 1),
    (2, 13),
    (3, 18),
    (4, 22),
    (5, 19),
    (6, 120),
    (7, 0),
    (8, 12),
    (9, 160 )
]

In [43]:
df_people = spark.createDataFrame(people, schema="id INT, age INT")
df_people.show()

+---+---+
| id|age|
+---+---+
|  1|  1|
|  2| 13|
|  3| 18|
|  4| 22|
|  5| 19|
|  6|120|
|  7|  0|
|  8| 12|
|  9|160|
+---+---+



In [46]:
df_people. \
    withColumn(
        "category",
        expr("""
        CASE
            WHEN age BETWEEN 0 AND 2 THEN "New Born"
            WHEN age > 2 AND age <= 12 THEN "Infant"
            WHEN age > 12 AND age <= 48 THEN "Toddler"
            WHEN age > 48 AND age <= 144 THEN "Kid"
            ELSE "Tenager or Adult"
        END
        """)
    ).\
    show()

+---+---+----------------+
| id|age|        category|
+---+---+----------------+
|  1|  1|        New Born|
|  2| 13|         Toddler|
|  3| 18|         Toddler|
|  4| 22|         Toddler|
|  5| 19|         Toddler|
|  6|120|             Kid|
|  7|  0|        New Born|
|  8| 12|          Infant|
|  9|160|Tenager or Adult|
+---+---+----------------+



In [52]:
(
    df_people
        .withColumn(
            "category",
            when(col("age").between(0, 2), "New Born")
            .when((col("age") > 2) & (col("age") <= 12), "Infant")
            .otherwise("another")
        )
        .show()
)

+---+---+--------+
| id|age|category|
+---+---+--------+
|  1|  1|New Born|
|  2| 13| another|
|  3| 18| another|
|  4| 22| another|
|  5| 19| another|
|  6|120| another|
|  7|  0|New Born|
|  8| 12|  Infant|
|  9|160| another|
+---+---+--------+

